In [1]:
!pip install rdkit-pypi xgboost -q

In [2]:
import pandas as pd

df = pd.read_csv('egfr_clean.csv')
print(df.shape)
df.head()

(607, 2)


,SMILES,pIC50
0,Brc1cccc(Nc2[nH]cnc3nc4ccccc4c2-3)c1,6.854649
1,Brc1cccc(Nc2ccnc3ccccc23)c1,5.259637
2,Brc1cccc(Nc2ncnc3c2[nH]c2ccccc23)c1.Cl,7.087092
3,Brc1cccc(Nc2ncnc3c2ccc2[nH]cnc23)c1,6.565431
4,Brc1cccc(Nc2ncnc3c2oc2ccccc23)c1,6.130768


In [3]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error

def get_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
    return np.array(fp)

df['fp'] = df['SMILES'].apply(get_fingerprint)
df_valid = df.dropna(subset=['fp'])
print("Valid molecules after fingerprinting:", df_valid.shape)

X = np.stack(df_valid['fp'].values)
y = df_valid['pIC50'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train size:", X_train.shape, "Test size:", X_test.shape)

Valid molecules after fingerprinting: (607, 3)
Train size: (485, 1024) Test size: (122, 1024)


In [4]:
# Random Forest (baseline, comparable to your original project's model type)
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_r2 = r2_score(y_test, rf_pred)
rf_mae = mean_absolute_error(y_test, rf_pred)
print(f"Random Forest + Fingerprints — R²: {rf_r2:.3f}, MAE: {rf_mae:.3f}")

# XGBoost (upgraded model)
xgb = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
xgb_r2 = r2_score(y_test, xgb_pred)
xgb_mae = mean_absolute_error(y_test, xgb_pred)
print(f"XGBoost + Fingerprints — R²: {xgb_r2:.3f}, MAE: {xgb_mae:.3f}")

Random Forest + Fingerprints — R²: 0.792, MAE: 0.505
XGBoost + Fingerprints — R²: 0.797, MAE: 0.500
